# 🎙️ VoxCPM2 Worker (Colab GPU Engine)

Worker alternatif untuk VoizGen Studio, memakai **VoxCPM2** (OpenBMB, 2B, Apache-2.0).
Kontrak HTTP-nya identik dengan `voizgenWorker.ipynb`, jadi web app yang sama bisa dipakai —
cukup ganti Tiket Koneksi / Kode Akses.

### Bedanya dari worker OmniVoice
| | OmniVoice | VoxCPM2 |
|---|---|---|
| Gaya suara | kosakata tertutup (`female`, `whisper`, …) | **teks bebas**: `energetic, shouting, cheerful` |
| Bahasa | 600+ | 30 (termasuk Indonesia) |
| Keluaran | 24 kHz | **48 kHz** |
| Ukuran | 0,6B | 2B (~8 GB VRAM) |
| Lisensi bobot | CC-BY-NC | **Apache-2.0** (boleh komersial) |

### Cara pakai
1. `Runtime` → `Change runtime type` → **T4 GPU** (atau lebih besar).
2. Isi form bila memakai gateway, lalu jalankan sel.
3. Klik **TAUTAN KONEKSI** di bawah banner — web app terbuka dan langsung terhubung.
   (Tanpa tautan: salin **TIKET KONEKSI** / **KODE AKSES** ke web app secara manual.)
4. Aktifkan `BENCHMARK` sekali untuk mengukur RTF di GPU yang sedang kamu pakai.


In [ ]:
#@title 🎙️ VoxCPM2 Worker Engine — klik Play
#@markdown Kosongkan kedua kolom bila web app dibuka langsung (mode tiket).
#@markdown Isi keduanya bila memakai VoizGen Gateway di Cloudflare Workers.
GATEWAY_URL = ""  #@param {type:"string"}
REGISTER_KEY = ""  #@param {type:"string"}
#@markdown Alamat web app-mu (mode tiket, mis. `https://voizgen.pages.dev`). Diisi →
#@markdown output memuat tautan sekali klik; dikosongkan → tiket saja untuk ditempel.
WEB_APP_URL = ""  #@param {type:"string"}
#@markdown Ukur RTF (real-time factor) di GPU ini sebelum melayani permintaan:
BENCHMARK = True  #@param {type:"boolean"}
#@markdown Muat denoiser ZipEnhancer (membersihkan audio referensi voice clone, unduhan tambahan):
LOAD_DENOISER = True  #@param {type:"boolean"}
#@markdown Kompilasi dengan `torch.compile` (warm-up ~1,5 menit di T4; matikan untuk start cepat):
OPTIMIZE = True  #@param {type:"boolean"}

!pip install -q voxcpm pycloudflared fastapi "uvicorn[standard]" python-multipart

import base64
import inspect
import io
import os
import re
import secrets
import tempfile
import threading
import time
import urllib.parse
import wave

import numpy as np
import requests
import torch
import uvicorn
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response
from pycloudflared import try_cloudflare

from voxcpm import VoxCPM

PORT = int(os.environ.get("VOIZGEN_PORT", 7860))
MODEL_ID = os.environ.get("VOIZGEN_MODEL", "openbmb/VoxCPM2")
TOKEN = os.environ.get("VOIZGEN_TOKEN") or secrets.token_urlsafe(18)

# VoxCPM2 mendukung 30 bahasa tanpa perlu tag bahasa; daftar ini hanya untuk
# ditampilkan di web app. Pilihan bahasa tidak dikirim ke model.
LANGUAGES = [
    "Arabic", "Burmese", "Chinese", "Danish", "Dutch", "English", "Finnish", "French",
    "German", "Greek", "Hebrew", "Hindi", "Indonesian", "Italian", "Japanese", "Khmer",
    "Korean", "Lao", "Malay", "Norwegian", "Polish", "Portuguese", "Russian", "Spanish",
    "Swahili", "Swedish", "Tagalog", "Thai", "Turkish", "Vietnamese",
]

# ---------------------------------------------------------------- model
IS_GPU = torch.cuda.is_available()
if not IS_GPU:
    print("\033[1;31m[PERINGATAN] Tidak ada GPU. VoxCPM2 2B di CPU sangat lambat; "
          "ubah ke Runtime > Change runtime type > T4 GPU.\033[0m", flush=True)
else:
    print(f"[VoxCPM2] GPU: {torch.cuda.get_device_name(0)}", flush=True)

print(f"[VoxCPM2] Memuat {MODEL_ID}...", flush=True)
if IS_GPU and not torch.cuda.is_bf16_supported():
    print("[VoxCPM2] GPU ini tidak mendukung bfloat16 native (mis. T4) — torch.compile "
          "akan melewati sebagian kernel. Ini normal, bukan error.", flush=True)

model = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=LOAD_DENOISER, optimize=OPTIMIZE)
HAS_DENOISER = LOAD_DENOISER

# Sample rate diambil dari model (48 kHz untuk VoxCPM2), bukan diasumsikan.
SAMPLING_RATE = int(getattr(getattr(model, "tts_model", None), "sample_rate", 0) or 16000)

def _generate_params(target):
    """Parameter yang benar-benar diterima generate().

    `VoxCPM.generate()` hanya meneruskan `*args/**kwargs` ke `_generate()`, jadi
    tanda tangannya kosong dan tidak bisa dipakai menyaring. Ambil dari
    implementasi sebenarnya; kalau keduanya buram, kembalikan None supaya
    argumen diteruskan apa adanya (biar TypeError-nya jujur, bukan disembunyikan).
    """
    for name in ("_generate", "generate"):
        fn = getattr(target, name, None)
        if fn is None:
            continue
        params = inspect.signature(fn).parameters
        named = {k for k, v in params.items()
                 if v.kind not in (v.VAR_POSITIONAL, v.VAR_KEYWORD) and k != "self"}
        if named:
            return named
    return None


GEN_PARAMS = _generate_params(model)
if GEN_PARAMS is not None and "reference_wav_path" not in GEN_PARAMS:
    raise RuntimeError(
        "Versi voxcpm terpasang tidak punya parameter 'reference_wav_path' "
        f"(yang terbaca: {sorted(GEN_PARAMS)}). Perlu voxcpm >= 2.0. "
        "Jalankan: pip install -U voxcpm"
    )
print(f"[VoxCPM2] Parameter generate(): "
      f"{'tidak terbaca -> tanpa penyaringan' if GEN_PARAMS is None else len(GEN_PARAMS)}", flush=True)

GPU_LOCK = threading.Lock()  # satu inferensi sekaligus

# Instruct VoxCPM2 adalah teks bebas dalam tanda kurung, bukan kosakata tertutup.
INSTRUCT_ATTRIBUTES: list[str] = []
INSTRUCT_FREEFORM = True
INSTRUCT_EXAMPLES = [
    "energetic, enthusiastic",
    "shouting, urgent",
    "calm and soothing",
    "whispering, mysterious",
    "cheerful, fast paced",
    "an elderly man, low pitch",
]
# VoxCPM2 tidak mendokumentasikan tag non-verbal; jangan janjikan yang tidak ada.
NONVERBAL_TAGS: list[str] = []
DIRECTIVES = ["pause", "pause:1.5", "slow", "fast", "whisper", "normal"]

# Arahan kecepatan/bisikan tidak punya parameter API di VoxCPM2 — dipetakan ke
# deskripsi gaya, yang justru cara resmi mengatur pace menurut dokumentasinya.
DIRECTIVE_STYLE = {
    "slow": "speaking slowly",
    "lambat": "speaking slowly",
    "fast": "speaking quickly",
    "cepat": "speaking quickly",
}
DIRECTIVE_WHISPER = {"whisper", "berbisik"}
DIRECTIVE_RESET = {"normal", "reset"}
DEFAULT_PAUSE = 0.4
MAX_PAUSE = 5.0
MAX_SEGMENTS = 40

_BRACKET_RE = re.compile(r"\[([^\[\]]*)\]")
_PAUSE_RE = re.compile(r"^(?:pause|jeda)(?:\s*[:=]\s*([0-9]*\.?[0-9]+))?$")


# ---------------------------------------------------------------- helper
def to_wav(samples: np.ndarray, sampling_rate: int) -> bytes:
    """float32 [-1, 1] -> WAV PCM 16-bit mono. Clip wajib: >1.0 akan wrap-around."""
    pcm = (np.clip(samples, -1.0, 1.0) * 32767.0).astype("<i2")
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(int(sampling_rate))
        wav.writeframes(pcm.tobytes())
    return buf.getvalue()


def require_token(authorization: str = Header(default="")) -> None:
    supplied = authorization.removeprefix("Bearer ").strip()
    if not secrets.compare_digest(supplied, TOKEN):
        raise HTTPException(status_code=401, detail="Token tidak valid.")


def parse_script(raw: str):
    """Pecah naskah berarah menjadi segmen + jeda.

    Segmen: {"text", "style", "pause_after"} — `style` adalah daftar frasa gaya
    yang akan digabung ke deskripsi dalam tanda kurung.
    """
    segments: list[dict] = []
    dropped: list[str] = []
    pace: str | None = None
    whisper = False
    buffer: list[str] = []
    pos = 0

    def flush(pause_after: float = 0.0) -> None:
        nonlocal buffer
        body = " ".join("".join(buffer).split())
        if body:
            style = [s for s in (pace, "whispering" if whisper else None) if s]
            segments.append({"text": body, "style": style, "pause_after": pause_after})
        elif pause_after and segments:
            segments[-1]["pause_after"] = min(
                segments[-1]["pause_after"] + pause_after, MAX_PAUSE
            )
        buffer = []

    for match in _BRACKET_RE.finditer(raw):
        inner = match.group(1).strip()
        low = inner.lower()
        buffer.append(raw[pos:match.start()])
        pos = match.end()

        pause = _PAUSE_RE.match(low)
        if pause:
            seconds = float(pause.group(1)) if pause.group(1) else DEFAULT_PAUSE
            flush(min(seconds, MAX_PAUSE))
        elif low in DIRECTIVE_STYLE:
            flush()
            pace = DIRECTIVE_STYLE[low]
        elif low in DIRECTIVE_WHISPER:
            flush()
            whisper = True
        elif low in DIRECTIVE_RESET:
            flush()
            pace, whisper = None, False
        else:
            dropped.append(inner)

    buffer.append(raw[pos:])
    flush()
    return segments, dropped


def build_prompt(text: str, style_parts: list[str]) -> str:
    """Gabungkan deskripsi gaya ke depan naskah sesuai konvensi VoxCPM2: "(gaya) teks"."""
    parts = []
    for part in style_parts:
        cleaned = part.strip().strip("()").strip()
        if cleaned and cleaned not in parts:
            parts.append(cleaned)
    return f"({', '.join(parts)}) {text}" if parts else text


def call_generate(**kwargs):
    """Kirim hanya argumen yang diterima versi voxcpm terpasang."""
    if GEN_PARAMS is not None:
        kwargs = {k: v for k, v in kwargs.items() if k in GEN_PARAMS}
    return model.generate(**kwargs)


def join_segments(waves, segments, sampling_rate: int) -> np.ndarray:
    pieces = []
    for wave_data, segment in zip(waves, segments):
        pieces.append(np.asarray(wave_data, dtype=np.float32).reshape(-1))
        gap = segment["pause_after"]
        if gap > 0:
            pieces.append(np.zeros(int(gap * sampling_rate), dtype=np.float32))
    return pieces[0] if len(pieces) == 1 else np.concatenate(pieces)


# ---------------------------------------------------------------- benchmark
BENCH_TEXT = (
    "Selamat pagi, ini pengukuran kecepatan sintesis suara pada perangkat ini "
    "menggunakan naskah bahasa Indonesia yang cukup panjang."
)
BENCH_RESULT: list[dict] = []


def run_benchmark() -> None:
    """RTF = waktu inferensi / durasi audio. <1 berarti lebih cepat dari real-time."""
    print("[VoxCPM2] Mengukur RTF...", flush=True)
    for timesteps in (10, 32):
        started = time.perf_counter()
        wav = call_generate(text=BENCH_TEXT, cfg_value=2.0, inference_timesteps=timesteps)
        elapsed = time.perf_counter() - started
        audio = np.asarray(wav, dtype=np.float32).reshape(-1)
        seconds = len(audio) / SAMPLING_RATE
        rtf = elapsed / seconds if seconds else float("nan")
        BENCH_RESULT.append({
            "inference_timesteps": timesteps,
            "audio_seconds": round(seconds, 2),
            "elapsed_seconds": round(elapsed, 2),
            "rtf": round(rtf, 3),
        })
        print(f"  timesteps={timesteps:>2}  audio={seconds:5.2f}s  "
              f"inferensi={elapsed:5.2f}s  RTF={rtf:.3f}", flush=True)


# ---------------------------------------------------------------- api
app = FastAPI(title="VoxCPM2 Worker", docs_url=None, redoc_url=None)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
    expose_headers=["X-Elapsed", "X-Segments", "X-Notes"],
    max_age=86400,
)


@app.get("/api/health", dependencies=[Depends(require_token)])
def health():
    return {
        "ok": True,
        "engine": "voxcpm2",
        "device": torch.cuda.get_device_name(0) if IS_GPU else "cpu",
        "gpu": IS_GPU,
        "dtype": "bfloat16" if IS_GPU else "float32",
        "model": MODEL_ID,
        "sampling_rate": SAMPLING_RATE,
        "languages": LANGUAGES,
        "instruct_attributes": INSTRUCT_ATTRIBUTES,
        "instruct_freeform": INSTRUCT_FREEFORM,
        # 10 timesteps adalah default vendor VoxCPM2 — memakai 32 seperti OmniVoice
        # berarti 3x biaya difusi tanpa alasan. UI mengikuti angka ini saat terhubung.
        "defaults": {"num_step": 10, "guidance_scale": 2.0},
        "instruct_examples": INSTRUCT_EXAMPLES,
        "nonverbal_tags": NONVERBAL_TAGS,
        "directives": DIRECTIVES,
        "benchmark": BENCH_RESULT,
        "denoiser": HAS_DENOISER,
        "compiled": OPTIMIZE,
        "busy": GPU_LOCK.locked(),
    }


@app.post("/api/generate", dependencies=[Depends(require_token)])
def generate(
    mode: str = Form("design"),
    text: str = Form(...),
    language: str = Form(""),
    instruct: str = Form(""),
    ref_text: str = Form(""),
    num_step: int = Form(10),  # default vendor VoxCPM2
    guidance_scale: float = Form(2.0),
    denoise: bool = Form(True),
    speed: float = Form(1.0),
    duration: float = Form(0.0),
    preprocess_prompt: bool = Form(True),
    postprocess_output: bool = Form(True),
    ref_audio: UploadFile | None = File(None),
):
    if not text.strip():
        raise HTTPException(status_code=400, detail="Naskah teks kosong.")
    if mode not in ("clone", "design"):
        raise HTTPException(status_code=400, detail=f"Mode tidak dikenal: {mode}")

    segments, dropped = parse_script(text)
    if not segments:
        raise HTTPException(status_code=400, detail="Naskah hanya berisi tag, tidak ada teks untuk diucapkan.")
    if len(segments) > MAX_SEGMENTS:
        raise HTTPException(
            status_code=400,
            detail=f"Naskah terpecah menjadi {len(segments)} segmen (maksimum {MAX_SEGMENTS}).",
        )

    notes = []
    if dropped:
        notes.append("tag tidak dikenali diabaikan: " + ", ".join(f"[{d}]" for d in dict.fromkeys(dropped)))
    if speed != 1.0:
        notes.append("VoxCPM2 tidak punya parameter kecepatan; pakai [slow]/[fast] atau tulis 'speaking slowly' di Gaya Suara")
    if duration > 0:
        notes.append("VoxCPM2 tidak mendukung durasi tetap")
    if language.strip():
        notes.append("pilihan bahasa diabaikan: VoxCPM2 mendeteksi bahasa dari teks")
    if denoise and not HAS_DENOISER:
        notes.append("denoise dilewati: worker dijalankan tanpa denoiser")

    ref_path = None
    try:
        if mode == "clone":
            if ref_audio is None:
                raise HTTPException(status_code=400, detail="Audio referensi wajib untuk voice clone.")
            suffix = os.path.splitext(ref_audio.filename or "")[1] or ".wav"
            with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as handle:
                handle.write(ref_audio.file.read())
                ref_path = handle.name

        base_style = instruct.strip()
        started = time.perf_counter()
        waves = []
        try:
            with GPU_LOCK:
                for segment in segments:
                    kwargs = {
                        "text": build_prompt(segment["text"], [base_style, *segment["style"]]),
                        "cfg_value": guidance_scale,
                        "inference_timesteps": max(1, num_step),
                        "denoise": denoise and HAS_DENOISER,
                        "normalize": preprocess_prompt,
                    }
                    if ref_path:
                        kwargs["reference_wav_path"] = ref_path
                        # Transkrip referensi -> mode gabungan (timbre + konteks),
                        # yang menurut dokumentasi sedikit menaikkan kemiripan.
                        if ref_text.strip():
                            kwargs["prompt_wav_path"] = ref_path
                            kwargs["prompt_text"] = ref_text.strip()
                    waves.append(call_generate(**kwargs))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            return JSONResponse({"detail": "VRAM habis. Kurangi panjang teks atau inference steps."}, 503)
        except Exception as exc:
            return JSONResponse({"detail": f"{type(exc).__name__}: {exc}"}, 500)
        elapsed = time.perf_counter() - started
    finally:
        if ref_path:
            os.unlink(ref_path)

    headers = {"X-Elapsed": f"{elapsed:.2f}", "X-Segments": str(len(segments))}
    if notes:
        headers["X-Notes"] = urllib.parse.quote("; ".join(notes))

    return Response(
        content=to_wav(join_segments(waves, segments, SAMPLING_RATE), SAMPLING_RATE),
        media_type="audio/wav",
        headers=headers,
    )


# ---------------------------------------------------------------- gateway
GATEWAY = (GATEWAY_URL or os.environ.get("VOIZGEN_GATEWAY", "")).strip().rstrip("/")
REG_KEY = (REGISTER_KEY or os.environ.get("VOIZGEN_REGISTER_KEY", "")).strip()
HEARTBEAT_EVERY = 300


def gateway_call(path: str, payload: dict, timeout: int = 15) -> dict:
    resp = requests.post(
        f"{GATEWAY}{path}",
        json=payload,
        headers={"X-Register-Key": REG_KEY},
        timeout=timeout,
    )
    if resp.status_code >= 400:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
    return resp.json()


def gateway_heartbeat_loop(code: str) -> None:
    while True:
        time.sleep(HEARTBEAT_EVERY)
        try:
            gateway_call("/api/heartbeat", {"code": code}, timeout=10)
        except Exception as exc:
            print(f"[VoxCPM2] Heartbeat gagal: {exc}", flush=True)


# ---------------------------------------------------------------- boot
if BENCHMARK:
    run_benchmark()

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.2)

tunnel = try_cloudflare(port=PORT)
public_url = str(getattr(tunnel, "tunnel", None) or getattr(tunnel, "url", None) or tunnel).rstrip("/")
ticket = "voizgen_" + base64.urlsafe_b64encode(f"{public_url}|{TOKEN}".encode()).decode().rstrip("=")

access_code = None
if GATEWAY and REG_KEY:
    try:
        result = gateway_call("/api/register", {"tunnel_url": public_url, "worker_token": TOKEN})
        access_code = result.get("pretty") or result["code"]
        threading.Thread(target=gateway_heartbeat_loop, args=(result["code"],), daemon=True).start()
    except Exception as exc:
        print(f"\033[1;31m[VoxCPM2] Registrasi ke gateway gagal: {exc}\033[0m", flush=True)
        print("Pakai TIKET KONEKSI di bawah sebagai gantinya.", flush=True)
elif GATEWAY or REG_KEY:
    print("\033[1;33m[VoxCPM2] GATEWAY_URL dan REGISTER_KEY harus diisi berdua.\033[0m", flush=True)

device_line = torch.cuda.get_device_name(0) if IS_GPU else "CPU (sangat lambat)"
bench_line = ""
if BENCH_RESULT:
    bench_line = "  RTF      : " + ", ".join(
        f"{b['rtf']:.2f} @ {b['inference_timesteps']} steps" for b in BENCH_RESULT
    ) + "\n"

# Tautan sekali klik. Kredensial ditaruh di fragment (#), bukan query: fragment
# tidak pernah dikirim ke server, jadi token/kode tak masuk log Cloudflare.
APP = (GATEWAY if access_code else (WEB_APP_URL or os.environ.get("VOIZGEN_APP", "")).strip()).rstrip("/")
if APP and access_code:
    quick_link = f"{APP}/#c={access_code}"
elif APP:
    quick_link = f"{APP}/#w={ticket}"
else:
    quick_link = ""

link_block = f"""  TAUTAN KONEKSI (klik — web app langsung terhubung):

  \033[1;36m{quick_link}\033[0m

------------------------------------------------------------
""" if quick_link else ""
if access_code:
    body = f"""{link_block}  KODE AKSES (bila menempel manual):

  \033[1;32m{access_code}\033[0m

  Web app  : \033[1;36m{GATEWAY}\033[0m"""
else:
    body = f"""{link_block}  TIKET KONEKSI (bila menempel manual):

  \033[1;32m{ticket}\033[0m

------------------------------------------------------------
  Manual, bila perlu:
  WORKER URL : \033[1;36m{public_url}\033[0m
  TOKEN      : \033[1;36m{TOKEN}\033[0m"""

print(f"""
============================================================
  VoxCPM2 Worker Engine  ·  {device_line}
============================================================
  Model    : {MODEL_ID}  ({SAMPLING_RATE} Hz)
  Gaya     : teks bebas, contoh "energetic, shouting"
  Opsi     : denoiser {'on' if HAS_DENOISER else 'off'} · torch.compile {'on' if OPTIMIZE else 'off'}
{bench_line}============================================================
{body}
============================================================
  Sel ini harus tetap berjalan. Interrupt untuk mematikan.
============================================================
""", flush=True)

# Colab tidak selalu menjadikan URL di stdout bisa diklik; anchor HTML selalu bisa.
if quick_link:
    try:
        from IPython.display import HTML, display

        display(HTML(
            f'<a href="{quick_link}" target="_blank" rel="noopener" style="display:inline-block;'
            'padding:10px 18px;margin:6px 0;border-radius:8px;background:#2563eb;color:#fff;'
            'font:600 14px system-ui,sans-serif;text-decoration:none">▶ Buka VoizGen Studio &amp; hubungkan</a>'
        ))
    except ImportError:
        pass

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    server.should_exit = True
    if access_code and GATEWAY and REG_KEY:
        try:
            gateway_call("/api/deregister", {"code": access_code.replace("-", "")}, timeout=10)
            print("[VoxCPM2] Kode akses dicabut dari gateway.")
        except Exception as exc:
            print(f"[VoxCPM2] Gagal mencabut kode: {exc}")
    print("[VoxCPM2] Worker dimatikan.")
